# Gold: patient_360

Builds `ws_healthcare_cntrl_india.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per STTM `sttm_silver_to_gold.xlsx` (sheet `patient_360`). Full rebuild with LEFT JOINs from patients so no patient is dropped.

In [0]:
%sql
CREATE OR REPLACE TABLE ws_healthcare_cntrl_india.gold.patient_360
COMMENT 'Gold patient 360: one row per patient with hospital, insurance and lab test summary. Source: STTM silver_to_gold, sheet patient_360.'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count,
    MAX(test_date)                            AS last_test_date
  FROM ws_healthcare_cntrl_india.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                              AS date_of_birth,  -- silver stores TIMESTAMP
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                   AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name                                                            AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                    AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                            AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                            AS critical_test_count,
  l.last_test_date
FROM ws_healthcare_cntrl_india.silver.patients p
LEFT JOIN ws_healthcare_cntrl_india.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN ws_healthcare_cntrl_india.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id;

In [0]:
# Return the gold table record count to the caller (e.g. a Job or dbutils.notebook.run)
updated_df = spark.table("ws_healthcare_cntrl_india.gold.patient_360")
record_count = updated_df.count()
print(f"gold.patient_360 record count: {record_count}")

dbutils.notebook.exit(str(record_count))